# Peak Month Distribution
Generates `peak_month_distribution.csv` for the Tableau dashboard.

**Input:** `mosquito_suitability.csv` — ERA5 climate normals, one row per city × month  
**Output:** `peak_month_distribution.csv` — long format, one row per species × month with city count  
**Logic:** Peak month = month with highest suitability score per city per species (moderate threshold not applied here — peak month is threshold-independent)

In [1]:
import pandas as pd

# Config
INPUT_FILE  = "mosquito_suitability.csv"   # adjust path if needed
OUTPUT_FILE = "peak_month_distribution.csv"

MONTH_NAMES = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}

In [2]:
# Load
df = pd.read_csv(INPUT_FILE)
print(f"Rows loaded: {len(df):,}")
print(f"Cities: {df['city'].nunique():,}")
df.head(3)

Rows loaded: 17,076
Cities: 1,418


,city,country,lat,lon,population,month,temp_mean_c,rh_mean_pct,precip_sum_mm,vpd_kpa,photoperiod_h,vpd_score,temp_score_aegypti,temp_score_albopictus,photo_factor_albopictus_temperate_only,suitability_score_aegypti,suitability_score_albopictus,elevation_m,special_interest
0,Tokyo,Japan,35.687,139.7495,37785000,1,4.85,62.3,70.2,0.3250,9.84,1.0,0.0,0.0,0.0023,0.0,0.0,32.0,0
1,Tokyo,Japan,35.687,139.7495,37785000,2,5.74,61.6,68.4,0.3523,10.70,1.0,0.0,0.0,0.0023,0.0,0.0,32.0,0
2,Tokyo,Japan,35.687,139.7495,37785000,3,9.14,63.4,113.3,0.4238,11.73,1.0,0.0,0.0,0.0023,0.0,0.0,32.0,0


In [3]:
# Peak month per city per species
# idxmax returns the index of the row with the highest suitability score
# per city/country group, that row's month is the peak month.

group_cols = ["city", "country"]

idx_aeg = df.groupby(group_cols)["suitability_score_aegypti"].idxmax()
idx_alb = df.groupby(group_cols)["suitability_score_albopictus"].idxmax()

peak_aeg = df.loc[idx_aeg, group_cols + ["month"]].copy()
peak_aeg["species"] = "Ae. aegypti"

peak_alb = df.loc[idx_alb, group_cols + ["month"]].copy()
peak_alb["species"] = "Ae. albopictus"

peaks = pd.concat([peak_aeg, peak_alb], ignore_index=True)
peaks.rename(columns={"month": "peak_month"}, inplace=True)

print(f"Peak month rows: {len(peaks):,} (should be {df['city'].nunique() * 2:,})")

Peak month rows: 2,846 (should be 2,836)


In [4]:
# Aggregate: city count per species × peak month
dist = (
    peaks
    .groupby(["species", "peak_month"])
    .size()
    .reset_index(name="city_count")
)

dist["month_name"] = dist["peak_month"].map(MONTH_NAMES)

# Ensure all 12 months are present for both species (fill missing with 0)
full_index = pd.MultiIndex.from_product(
    [dist["species"].unique(), range(1, 13)],
    names=["species", "peak_month"]
)
dist = (
    dist.set_index(["species", "peak_month"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)
dist["month_name"] = dist["peak_month"].map(MONTH_NAMES)

dist.sort_values(["species", "peak_month"], inplace=True)
dist

,species,peak_month,city_count,month_name
0,Ae. aegypti,1,45,Jan
1,Ae. aegypti,2,50,Feb
2,Ae. aegypti,3,27,Mar
3,Ae. aegypti,4,42,Apr
4,Ae. aegypti,5,92,May
5,Ae. aegypti,6,127,Jun
6,Ae. aegypti,7,402,Jul
7,Ae. aegypti,8,338,Aug
8,Ae. aegypti,9,128,Sep
9,Ae. aegypti,10,83,Oct


In [5]:
# Sanity check
totals = dist.groupby("species")["city_count"].sum()
print("Total cities per species (should both equal total cities in dataset):")
print(totals)
assert totals.nunique() == 1, "City counts differ between species — check input data"

Total cities per species (should both equal total cities in dataset):
species
Ae. aegypti       1423
Ae. albopictus    1423
Name: city_count, dtype: int64


In [6]:
#  Save
dist.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")
print(f"Rows: {len(dist)}  |  Columns: {list(dist.columns)}")

Saved: peak_month_distribution.csv
Rows: 12  |  Columns: ['peak_month', 'month_name', 'city_count_aegypti', 'city_count_albopictus']
